In [2]:
import pandas as pd
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv
 

In [3]:
load_dotenv()
engine = create_engine(
    f"postgresql+psycopg2://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}/{os.getenv('DB_NAME')}"
)


Loading Master Dataset

In [6]:
query = """
SELECT 
  f.value,
  t.data_period,
  h.hospital_name, h.state, h.hospital_type,
  m.measure_code, m.measure_name, m.unit,
  rm.reported_measure_name

FROM warehouse.fact_measure_performance f
LEFT JOIN warehouse.dim_hospital h ON f.hospital_id = h.hospital_id
LEFT JOIN warehouse.dim_measure m ON f.measure_id = m.measure_id
LEFT JOIN warehouse.dim_reported_measure rm ON f.reported_measure_id = rm.reported_measure_id
LEFT JOIN warehouse.dim_time t ON f.time_id = t.time_id

  """

df = pd.read_sql(query, engine)


Basic Cleaning

In [7]:
# dropping rows with missing critical missing values
df = df.dropna(subset=['value', 'hospital_name', 'measure_name', 'hospital_type'])
# stripping text fields and standardizing case title
for col in ['hospital_name', 'state', 'hospital_type', 'measure_code', 'measure_name', 'unit', 'reported_measure_name']:
  df[col]= df[col].str.strip().str.title()
# Ensuring value is numeric through coercion
df['value'] = pd.to_numeric(df['value'], errors='coerce')
df = df.dropna(subset=['value'])
# removing duplicates
df = df.drop_duplicates()
